# Check emission inventories

In [1]:
# import 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from pathlib import Path

from matplotlib.colors import LogNorm, SymLogNorm

import glob

import cmcrameri.cm as cmc

In [2]:
def read_edgar_files(dir_data, file_name, emission_source, yr1, yr2, return_mean=False):

    ## Read all yearly inventories and save as a dict
    print(f"for year {yr1} to {yr2}")
    file_pattern = f"{dir_data}/{file_name}*_{emission_source}_flx.nc"
    print("Files:", file_pattern)

    # Get a sorted list of filenames
    file_list = sorted(glob.glob(file_pattern))

    # Open datasets separately, assigning a new time coordinate
    years = range(yr1,yr2+1)
    inventory_years = {}
    for file, year in zip(file_list, years):
        ds = xr.open_dataset(file)  # Open dataset
        ds = ds.expand_dims({'time': [pd.Timestamp(f"{year}-01-01")]})  # Add time dimension
        #datasets.append(ds)
        inventory_years[year] = ds
    if not inventory_years:
        print("No inventory files found!")
    else:
        return inventory_years if return_mean == False else xr.concat(list(inventory_years.values()), dim='time').mean(dim='time')

## EDGAR CO

In [3]:

## Read in EDGAR CO data
dir_data_edgar_co = "/input/EDGAR/v8.1/CO" # path to monthly fire data (here on ddm)
edgar_file_name = "v8.1_FT2022_AP_CO"

edgar_co_total = read_edgar_files(dir_data_edgar_co,edgar_file_name, "TOTALS", inventory_year, inventory_year, return_mean=True)
edgar_co_agri = read_edgar_files(dir_data_edgar_co,edgar_file_name, "AWB", inventory_year, inventory_year, return_mean=True) # AWB = Agricultural waste burning
edgar_co_total_no_awb = edgar_co_total - edgar_co_agri


NameError: name 'inventory_year' is not defined

In [ ]:
#### READ and export EDGAR CO Data

# Path to your NetCDF files (adjust if needed)
file_pattern = "/input/EDGAR/v8.1/CO/v8.1_FT2022_AP_CO_*_TOTALS_flx.nc"

# Get a sorted list of filenames
file_list = sorted(glob.glob(file_pattern))

# Extract years from filenames
years = [int(f.split("_")[4]) for f in file_list]  # Extract the year from filename

# Open datasets separately, assigning a new time coordinate
datasets = []
for file, year in zip(file_list, years):
    ds = xr.open_dataset(file)  # Open dataset
    ds = ds.expand_dims({'time': [pd.Timestamp(f"{year}-01-01")]})  # Add time dimension
    datasets.append(ds)

# Merge datasets along the new time axis
edgar = xr.concat(datasets, dim="time")

#edgar.sel(time=slice('2020-01-01','2022-01-01')).to_netcdf("/project/leob/GAW/Kenya/EDGAR/CO_2020_2022_TOTALS_flx.nc")

In [ ]:

plt.figure()
vmax = 1e-10
vmin = 1e-20
edgar.isel(time=-1).fluxes.plot(norm=LogNorm(vmin=vmin, vmax=vmax), cmap='viridis')
plt.show()

## WetCHARTs CH4 wetland data
Daily wetland emission data

In [ ]:
import re
file_pattern = "/project/leob/GAW/Kenya/data/WetCHARTs/WetCHARTs_v1_3_3_*.nc"

# Get a sorted list of filenames
file_list = sorted(glob.glob(file_pattern))

# Filter files to ensure they contain only a single 4-digit year
filtered_files = [f for f in file_list if re.search(r'/project/leob/GAW/Kenya/data/WetCHARTs/WetCHARTs_v1_3_3_\d{4}\.nc$', f)]

# Extract years from filenames
years = [int(re.search(r'(\d{4})\.nc$', f).group(1)) for f in filtered_files]

# Extract years from filenames
#years = [int(f.split("_")[-1][:-3]) for f in file_list]  # Extract the year from filename

# Open datasets separately, assigning a new time coordinate
datasets = []
for file, year in zip(filtered_files, years):
    ds = xr.open_dataset(file)  # Open dataset
    datasets.append(ds)

# Merge datasets along the new time axis
wetcharts = xr.concat(datasets, dim="time")
wetcharts

In [ ]:
wetcharts

In [ ]:
wetcharts_model_mean = wetcharts['wetland_CH4_emissions'].mean(dim='model')
wetcharts_model_mean
#wetcharts_model_mean.to_netcdf("/project/leob/GAW/Kenya/data/WetCHARTs/WetCHARTs_v1_3_3_2020_2022_model_mean.nc")

In [ ]:
# mean of all 18 model ensemble members
plt.figure()
(wetcharts_model_mean.mean(dim='lat').mean(dim='lon') * 1e-6 / (24*3600)).plot() #convert to kg/m2/s
plt.show()

In [ ]:
plt.figure()

#max = 1e-2
vmin = 1e-15
(wetcharts_model_mean.isel(time=7) * 1e-6 / (24*3600)).plot(norm=LogNorm(vmin=vmin), cmap='viridis')
plt.show()

#### regridding....

In [ ]:
# try to regrid edgar to a coarser 0.5x0.5 grid (but this takes super long)
ds1 = edgar
ds1
# Extract target grid from ds2
ds_out_grid = xr.Dataset(
    {
        "lat": (["lat"], wetcharts_model_mean["lat"].values),  # Use lat from wetcharts
        "lon": (["lon"], wetcharts_model_mean["lon"].values)   # Use lon from wetcharts
    }
)

# Assign each lat/lon point to a bin
ds1["lat_bins"] = xr.DataArray(np.digitize(ds1["lat"], ds_out_grid["lat"]) - 1, dims="lat")
ds1["lon_bins"] = xr.DataArray(np.digitize(ds1["lon"], ds_out_grid["lon"]) - 1, dims="lon")

# Group by the bins and sum emissions to preserve totals
ds_coarse = ds1.groupby(["lat_bins", "lon_bins"]).sum()

# Assign new lat/lon coordinates
ds_coarse = ds_coarse.assign_coords(
    lat=("lat_bins", lat_bins[ds_coarse["lat_bins"].values]),
    lon=("lon_bins", lon_bins[ds_coarse["lon_bins"].values])
).drop_vars(["lat_bins", "lon_bins"])


## EDGAR CH4 data

In [ ]:
## Read in EDGAR CH4 data
dir_data_edgar_ch4 = "/project/leob/GAW/Kenya/data/EDGAR/CH4_TOTALS_flx" # path to monthly fire data (here on ddm)
edgar_file_name = "EDGAR_2024_GHG_CH4"

edgar_ch4_total = read_edgar_files(dir_data_edgar_ch4,edgar_file_name, "TOTALS", inventory_year, inventory_year, return_mean=True)


In [ ]:
#### READ and export EDGAR CH4 Data

# Path to your NetCDF files (adjust if needed)
file_pattern = "/project/leob/GAW/Kenya/data/EDGAR/CH4_TOTALS_flx/EDGAR_2024_GHG_CH4_*.nc"

# Get a sorted list of filenames
file_list = sorted(glob.glob(file_pattern))

# Extract years from filenames
years = [int(f.split("_")[-3]) for f in file_list] # Extract the year from filename

# Open datasets separately, assigning a new time coordinate
datasets = []
for file, year in zip(file_list, years):
    ds = xr.open_dataset(file)  # Open dataset
    ds = ds.expand_dims({'time': [pd.Timestamp(f"{year}-01-01")]})  # Add time dimension
    datasets.append(ds)

# Merge datasets along the new time axis
edgar_ch4 = xr.concat(datasets, dim="time")

#edgar_ch4.to_netcdf("/project/leob/GAW/Kenya/EDGAR/CH4_{years[0]}_{years[-1]}_TOTALS_flx.nc")

In [ ]:
# mean of whole world
plt.figure()
edgar_ch4['fluxes'].mean(dim='lat').mean(dim='lon').plot()
plt.show()

In [ ]:

from matplotlib.colors import LogNorm, SymLogNorm
plt.figure()
vmax = 1e-9
vmin = 1e-15
edgar_ch4.fluxes.isel(time=-1).plot(norm=LogNorm(vmin=vmin,vmax=vmax), cmap='viridis')
plt.show()

## Load with emiproc

In [ ]:
import emiproc
from pathlib import Path
from emiproc.inventories.edgar import download_edgar_files

# check which emiproc version is used
print(emiproc.__file__)

In [ ]:
from pathlib import Path
from emiproc.inventories.edgar import download_edgar_files

local_dir = Path("/project/leob/GAW/Kenya/data/EDGAR/edgar")

local_dir.mkdir(exist_ok=True)


In [ ]:
# Download required EDGAR emission files (with emiproc its emissions, not fluxes!)
year_edgar = 2020
#download_edgar_files(local_dir, year=year_edgar, substances=["CH4"])

In [ ]:
from emiproc.inventories.edgar import EDGARv8

# Load the edgar inventory
# local_dir = Path("/project/leob/GAW/Kenya/EDGAR/CH4_TOTALS_flx/")
# filenames = "EDGAR_2024_GHG_CH4_*.nc"
#local_dir = Path("/input/EDGAR/v8.0_FT2022_GHG")
filenames = "v8.0_*.nc"
inv = EDGARv8(local_dir / filenames)

In [ ]:
# some info about the inventory
print(inv.grid)

In [ ]:
# We can look at the total emissions of the inventory (units are in kg/year)
inv.total_emissions.T



In [ ]:
import matplotlib.pyplot as plt
from emiproc.plots import plot_inventory

plt.style.use("ggplot")

plot_inventory(inv, total_only=True)



In [ ]:
## Remap the inventory to flexpart grid
from emiproc.grids import RegularGrid

# The grid can be defined by various parameters
# See the documentation for more details
# https://emiproc.readthedocs.io/en/master/api/grids.html#emiproc.grids.RegularGrid
african_grid = RegularGrid(xmin=0.05, xmax=60.05, ymin=-34.05, ymax=20.05, dx=0.1, dy=0.1) # 0.1 degree grid used in flexpart for Africa
african_grid

# Remap the inventory on the grid
from emiproc.regrid import remap_inventory

remapped = remap_inventory(inv, african_grid)
remapped


In [ ]:
plot_inventory(remapped, total_only=True, cmap = "cmc.davos_r")



In [ ]:
remapped.categories

In [ ]:
from emiproc.inventories.utils import group_categories

grouped = group_categories(
    remapped,
    categories_group={
        "agriculture": [
            "Agricultural soils",
            "Agricultural waste burning",
            "Manure management",
        ],
        "industry": [
            "Chemical processes",
            "Power Industry",
            "Oil refineries and Transformation industry",
            "Fuel exploitation",
            "Energy for buildings",
            "Combustion for manufacturing",
            "Iron and steel production",
            #"Non energy use of fuels", 
            #"Solvents and products use",
            #"Non-ferrous metals production",
            #"Non-metallic minerals production",
        ],
        "livestock": ["Enteric fermentation"],
        "waste": [
            "Waste water handling",
            "Solid waste incineration",
            "Solid waste landfills",
        ],
        "transportation": [
            "Aviation climbing_and_descent",
            "Aviation cruise",
            "Aviation landing_and_takeoff",
            "Railways, pipelines, off-road transport",
            "Shipping",
            "Road transportation",
        ],
    },
)


In [ ]:
for yr in years:
    local_dir = data_dir / f"EDGAR/{yr}"
    
    ## Reading in the downloaded files as inventories
    # Wetcharts has to be downloaded separately
    inv_edgar_ch4 = EDGARv8(local_dir / f"EDGAR_2024_GHG_CH4_{yr}_*.nc")

    inv_edgar_co = EDGARv8(local_dir / f"v8.1_FT2022_AP_CO_{yr}*.nc")

    inv_edgar_bc = EDGARv8(local_dir / f"v8.1_FT2022_AP_BC_{yr}*.nc")

    # Wetcharts: 
    wetchart_path = Path("/project/leob/GAW/Kenya/data/WetCHARTs/")
    file_path =  wetchart_path / f"WetCHARTs_v1_3_3_{yr}.nc"
    inv_wetcharts_ch4 = WetCHARTs(
        file_path,
        model=None, #use mean of all models
        category="wetland_emissions",
    )

    ### Remap the inventory on the grid
    inv_edgar_ch4_africa = remap_inventory(inv_edgar_ch4, african_grid)
    inv_edgar_co_africa = remap_inventory(inv_edgar_co, african_grid)
    inv_edgar_bc_africa = remap_inventory(inv_edgar_bc, african_grid)
    inv_wetcharts_ch4_africa = remap_inventory(inv_wetcharts_ch4, african_grid)

    ### Group the categories
    inv_edgar_ch4_africa_grouped = group_categories(inv_edgar_ch4_africa, categories_group = ghg_categories)
    inv_edgar_co_africa_grouped = group_categories(inv_edgar_co_africa, categories_group = ap_categories)
    inv_edgar_bc_africa_grouped = group_categories(inv_edgar_bc_africa, categories_group = ap_categories)

    ### Export the remapped and grouped inventories
    export_raster_netcdf(inv_edgar_ch4_africa_grouped, export_path / f"CH4_{yr}.nc")
    export_raster_netcdf(inv_edgar_co_africa_grouped, export_path / f"CO_{yr}.nc")
    export_raster_netcdf(inv_edgar_bc_africa_grouped, export_path / f"BC_{yr}.nc")

    # For wetcharts, export monthly files
    monthly_dir = wetchart_path / "monthly"
    monthly_dir.mkdir(parents=True, exist_ok=True)
    # saves monthly netcdfs in monthly_dir
    export_hourly_emissions(
        inv_wetcharts_ch4_africa,
        path=monthly_dir,
        freq="MS", # Start of the month
    )

In [ ]:
plot_inventory(grouped, total_only=True, cmap="viridis")

In [ ]:
## Export the remapped inventory to a NetCDF file
from emiproc.exports.rasters import export_raster_netcdf

export_path = local_dir / f"EDGAR_CH4_all_emissions_africa_{year_edgar}.nc"
export_raster_netcdf(remapped, export_path)
print("Exported")



## wetcharts with emiproc

In [ ]:

from emiproc.inventories.wetcharts import WetCHARTs
from emiproc.plots import plot_inventory
from emiproc import FILES_DIR
import xarray as xr


# Download the file and change the path here or anywhere you want.
wetchart_path = Path("/project/leob/GAW/Kenya/data/WetCHARTs/")
file_path =  wetchart_path / "WetCHARTs_v1_3_3_2021.nc"

In [ ]:
invCH4 = WetCHARTs(
    file_path,
    # Specify the model number to select from the dataset.
    model=None, #use mean of all models
    # Set the name of the category (default is "wetcharts").
    category="wetland_emissions",
)

In [ ]:
# some info about the inventory
print(invCH4.grid)
invCH4.t_profiles_groups 

In [ ]:
plot_inventory(invCH4, total_only=True)

In [ ]:
## Remap the inventory to flexpart grid
from emiproc.grids import RegularGrid

# The grid can be defined by various parameters
# See the documentation for more details
# https://emiproc.readthedocs.io/en/master/api/grids.html#emiproc.grids.RegularGrid
african_grid = RegularGrid(xmin=0.05, xmax=60.05, ymin=-34.05, ymax=20.05, dx=0.1, dy=0.1) # 0.1 degree grid used in flexpart for Africa
african_grid

# Remap the inventory on the grid
from emiproc.regrid import remap_inventory

remappedCH4 = remap_inventory(invCH4, african_grid)
remappedCH4

In [ ]:
remappedCH4.grid

In [ ]:
plot_inventory(remappedCH4, total_only=True)

In [ ]:
## get temporal scaling of wetcharts
from emiproc.exports.utils import get_temporally_scaled_array

da = get_temporally_scaled_array(invCH4, time_range=invCH4.year, sum_over_cells=False)

da.sel(category='wetland_emissions', substance='CH4').sum(dim='cell').plot()

In [ ]:
da.sel(category='wetland_emissions', substance='CH4')